# Phase 3 — trade imbalance (rebuilt)

See `STRATEGY.md` for the full 6-phase plan and the postmortem on this notebook's first attempt.
That attempt merged trade ticks with the Phase 1/2 minute-bar cache by `(ticker, minute)` — but
the two caches were fetched independently and ended up covering different calendar windows per
ticker, leaving usable overlap for only 11 of 28 tickers over roughly one day. **This rebuild
fixes that at the root**: instead of merging two independently-fetched sources, every feature
below — reversal score, VWAP distance, relative strength, and the new trade-imbalance (TOI)
feature — is derived directly from the same tick data, for every (ticker, minute) with at least
one trade. There is nothing left to misalign.

**Classification method, deliberately the simpler one**: each trade tick is classified by the
**tick rule** (an uptick from the previous trade is a buy, a downtick is a sell, an unchanged
price inherits the previous classification) rather than Lee-Ready (which needs matching quote
data too). See `download_trade_ticks.py`'s docstring for why.

```
TOI(minute) = (buy_volume - sell_volume) / (buy_volume + sell_volume)
```

**Scope**: all 126 successfully-downloaded tickers (5 skipped — see `SKIP_TICKERS` in
`download_trade_ticks.py`). The tick cache was built incrementally between 2026-10-07 and
2026-10-19 (each ticker fetched "most recent ~1.5 days" relative to whenever its own fetch ran),
so different tickers' data land on different calendar dates — 3 tickers on 2026-10-07 through 62
on 2026-10-19 (see the per-ticker `fetch_day` distribution in the first code cell's output).
Deriving every feature from each ticker's own tick data makes this a non-issue for everything
*within* a ticker, but it means the sample is **heavily weighted toward whichever few days most
tickers happened to be fetched on**, not a uniform one-month sample like Phase 1/2 — a real
difference to keep in mind when comparing magnitudes, not just a data-plumbing detail.

**Feature carried over from Phase 2, one dropped**: `vwap_dist` and `relative_strength` both
translate directly to tick-derived bars (within-session, well-posed regardless of how many
calendar days a ticker's cache spans). `volume_surprise` does **not** — it needs a baseline of
"this ticker's typical volume at this minute-of-session," which requires multiple comparable
sessions per ticker; with only ~1.5 days of tick history per ticker, that baseline would be
built from a single (or barely two) session's own data, which isn't a meaningful "surprise"
comparison. It's excluded here rather than computed in a way that's guaranteed to be close to
degenerate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICK_DATA_DIR = Path('data') / 'ticks'

K, H = 1, 1   # same cell Phase 1/2 found strongest
N_BUCKETS = 3
COST_BPS_GRID = [0, 1, 2, 5, 10]
TOP_FRACTION = 0.1

tick_tickers = sorted(p.stem for p in TICK_DATA_DIR.glob('*.csv'))
fetch_day = pd.Series({
    t: pd.read_csv(TICK_DATA_DIR / f'{t}.csv', usecols=['time'], parse_dates=['time'])['time'].max().date()
    for t in tick_tickers
})
print(f'{len(tick_tickers)} tickers with tick data')
print(fetch_day.value_counts().sort_index())

126 tickers with tick data
2026-10-07     3
2026-10-08     7
2026-10-09    17
2026-10-15    10
2026-10-16    27
2026-10-19    62
Name: count, dtype: int64


## Build 1-minute bars and features directly from tick data

For each ticker: classify every tick's direction by the tick rule, floor timestamps to the
minute, and aggregate into `close` (last trade price), `avg` (trade-size-weighted average price,
used as the running VWAP input), `volume`, and `toi`. `reversal_score`, `vwap_dist`, and the
forward return are all computed the same way Phase 1/2 defined them, just sourced from these
tick-derived bars instead of IBKR's own minute bars. Everything is grouped by `session_date`
(the ticker's own local trading-day boundary) so no feature or return ever crosses an overnight
gap.

In [2]:
def load_and_classify_ticks(ticker):
    frame = pd.read_csv(TICK_DATA_DIR / f'{ticker}.csv', parse_dates=['time'])
    frame['time'] = pd.to_datetime(frame['time'], utc=True)
    frame = frame.sort_values('time').reset_index(drop=True)
    # Tick rule: an uptick is a buy, a downtick is a sell, unchanged price inherits the
    # previous classification. The first tick with no prior price defaults to a buy (fillna(1.0)).
    diff = frame['price'].diff()
    frame['direction'] = np.sign(diff).replace(0, np.nan).ffill().fillna(1.0)
    frame['minute'] = frame['time'].dt.floor('min')
    frame['session_date'] = frame['time'].dt.tz_convert('America/New_York').dt.date
    return frame


def ticks_to_minute_bars(ticks):
    close = ticks.groupby('minute')['price'].last()
    vwap = (ticks['price'] * ticks['size']).groupby(ticks['minute']).sum() / ticks.groupby('minute')['size'].sum()
    volume = ticks.groupby('minute')['size'].sum()
    buy_vol = ticks.loc[ticks['direction'] > 0].groupby('minute')['size'].sum()
    sell_vol = ticks.loc[ticks['direction'] < 0].groupby('minute')['size'].sum()
    bars = pd.DataFrame({'close': close, 'avg': vwap, 'volume': volume,
                          'buy_vol': buy_vol, 'sell_vol': sell_vol}).fillna(0.0)
    bars.loc[close.index, 'close'] = close
    bars['toi'] = (bars['buy_vol'] - bars['sell_vol']) / (bars['buy_vol'] + bars['sell_vol']).replace(0, np.nan)
    bars['session_date'] = ticks.groupby('minute')['session_date'].first()
    return bars


frames = []
for ticker in tick_tickers:
    ticks = load_and_classify_ticks(ticker)
    bars = ticks_to_minute_bars(ticks)
    g = bars.groupby('session_date')
    trail = g['close'].transform(lambda s: s / s.shift(K) - 1)
    fwd = g['close'].transform(lambda s: s.shift(-H) / s - 1)
    reversal_score = -trail   # re-oriented, same convention as Phase 1/2: higher score -> higher predicted fwd return

    pv = bars['avg'] * bars['volume']
    cum_pv = pv.groupby(bars['session_date']).cumsum()
    cum_vol = bars['volume'].groupby(bars['session_date']).cumsum()
    vwap_dist = bars['close'] / (cum_pv / cum_vol) - 1

    frames.append(pd.DataFrame({
        'ticker': ticker, 'trail': trail, 'fwd': fwd, 'reversal_score': reversal_score,
        'toi': bars['toi'], 'vwap_dist': vwap_dist,
    }, index=bars.index))

panel = pd.concat(frames).dropna(subset=['reversal_score', 'fwd', 'toi', 'vwap_dist'])

# Cross-sectional relative strength: a z-score against every other ticker trading in that exact
# same minute. Unlike Phase 1/2's uniform one-month sample, most minutes here only have 1-2
# tickers present (since tickers' tick windows land on different calendar days) -- the z-score
# is only defined (non-NaN) where 2+ tickers genuinely share a minute, which dropna() below
# handles naturally rather than needing an explicit same-day-cohort grouping.
cs_mean = panel.groupby(panel.index)['trail'].transform('mean')
cs_std = panel.groupby(panel.index)['trail'].transform('std')
panel['relative_strength'] = (panel['trail'] - cs_mean) / cs_std

print(f'{len(tick_tickers)} tickers, {len(panel):,} (ticker, minute) observations')
print(f'date range: {panel.index.min()} to {panel.index.max()}')
print(f'{panel["relative_strength"].notna().sum():,} observations also have a defined relative_strength '
      f'(genuine same-minute overlap with at least one other ticker)')

126 tickers, 63,286 (ticker, minute) observations
date range: 2026-10-06 16:48:00+00:00 to 2026-10-19 19:58:00+00:00
63,277 observations also have a defined relative_strength (genuine same-minute overlap with at least one other ticker)


## Does trade imbalance predict the reversal?

Same method as Phase 2: bucket by the magnitude of each feature into terciles (`toi` and
`vwap_dist` can be negative, so bucket on `abs()`) and re-measure IC within each bucket against
the unconditional baseline.

In [3]:
baseline_ic = panel['reversal_score'].corr(panel['fwd'])
print(f'unconditional IC (tick-derived bars, k=h=1): {baseline_ic:+.4f}  (n={len(panel):,})')

for feat in ['toi', 'vwap_dist']:
    buckets = pd.qcut(panel[feat].abs(), N_BUCKETS, labels=['low', 'mid', 'high'], duplicates='drop')
    print(f'\n--- {feat} ---')
    for label, group in panel.groupby(buckets, observed=True):
        print(f'  {label}: n={len(group):,}, IC={group["reversal_score"].corr(group["fwd"]):+.4f}')

rs_panel = panel.dropna(subset=['relative_strength'])
buckets = pd.qcut(rs_panel['relative_strength'].abs(), N_BUCKETS, labels=['low', 'mid', 'high'], duplicates='drop')
print(f'\n--- relative_strength (n={len(rs_panel):,} observations with a defined z-score) ---')
for label, group in rs_panel.groupby(buckets, observed=True):
    print(f'  {label}: n={len(group):,}, IC={group["reversal_score"].corr(group["fwd"]):+.4f}')

unconditional IC (tick-derived bars, k=h=1): +0.2813  (n=63,286)

--- toi ---
  low: n=21,095, IC=+0.4005
  mid: n=21,095, IC=+0.2163
  high: n=21,096, IC=+0.1566

--- vwap_dist ---
  low: n=21,095, IC=+0.1440
  mid: n=21,096, IC=+0.0746
  high: n=21,095, IC=+0.4811

--- relative_strength (n=63,277 observations with a defined z-score) ---
  low: n=21,092, IC=+0.0126
  mid: n=21,092, IC=+0.0529
  high: n=21,093, IC=+0.3785


## Trade density sanity check

Tick-derived closes bounce between whichever side of the bid/ask the last trade happened to hit
— a thinly-traded minute (few trades) can show a much larger mechanical bid-ask-bounce reversal
than a genuinely liquid one. Check trade density before trusting the IC above as a real
predictive effect rather than a microstructure-noise artifact.

In [4]:
trade_counts = []
for ticker in tick_tickers:
    ticks = load_and_classify_ticks(ticker)
    trade_counts.append(ticks.groupby('minute').size())
trade_counts = pd.concat(trade_counts)
print(f'trades per minute across all {len(tick_tickers)} tickers: '
      f'median={trade_counts.median():.0f}, 25th pct={trade_counts.quantile(.25):.0f}, '
      f'10th pct={trade_counts.quantile(.10):.0f}')

trades per minute across all 126 tickers: median=38, 25th pct=27, 10th pct=19


## Combined filter: does trade imbalance add to Phase 2's filter?

Compare the original Phase 2 filter (`vwap_dist` + `relative_strength`, both in their high
tercile — `volume_surprise` dropped here since it isn't well-posed on this data, see intro) to
that same filter with `toi` added, the same "all features in their high tercile at once" test
Phase 2 used.

In [5]:
PHASE2_FEATURES = ['vwap_dist', 'relative_strength']   # volume_surprise excluded, see intro
ALL_FEATURES = PHASE2_FEATURES + ['toi']


def combined_filter_mask(data, features):
    mask = pd.Series(True, index=data.index)
    for feat in features:
        mask &= data[feat].abs() >= data[feat].abs().quantile(2 / 3)
    return mask


filtered_phase2 = rs_panel[combined_filter_mask(rs_panel, PHASE2_FEATURES)]
filtered_with_toi = rs_panel[combined_filter_mask(rs_panel, ALL_FEATURES)]

unfiltered_ic = rs_panel['reversal_score'].corr(rs_panel['fwd'])
phase2_ic = filtered_phase2['reversal_score'].corr(filtered_phase2['fwd'])
toi_ic = filtered_with_toi['reversal_score'].corr(filtered_with_toi['fwd'])
print(f'unconditional: n={len(rs_panel):,}, IC={unfiltered_ic:+.4f}')
print(f'vwap_dist + relative_strength filter: n={len(filtered_phase2):,} '
      f'({len(filtered_phase2)/len(rs_panel):.1%}), IC={phase2_ic:+.4f}')
print(f'+ toi filter: n={len(filtered_with_toi):,} '
      f'({len(filtered_with_toi)/len(rs_panel):.1%}), IC={toi_ic:+.4f}')

unconditional: n=63,277, IC=+0.2812
vwap_dist + relative_strength filter: n=8,480 (13.4%), IC=+0.5951
+ toi filter: n=2,884 (4.6%), IC=+0.2904


## Backtest: unconditional vs. filtered

Same long-short decile construction as Phase 1/2: every minute, go long the top decile of
`reversal_score` and short the bottom decile, average the forward-return spread across all
qualifying minutes (requires at least `min_names` names present that minute).

In [6]:
def decile_backtest(data, min_names=5):
    results = []
    for minute, group in data.groupby(data.index):
        if len(group) < min_names:
            continue
        n_decile = max(1, len(group) // 10)
        ranked = group.sort_values('reversal_score')
        longs = ranked.iloc[-n_decile:]
        shorts = ranked.iloc[:n_decile]
        results.append({'minute': minute, 'spread': longs['fwd'].mean() - shorts['fwd'].mean(), 'n': len(group)})
    return pd.DataFrame(results)


unconditional_bt = decile_backtest(panel, min_names=5)
filtered_phase2_bt = decile_backtest(filtered_phase2, min_names=5)
filtered_with_toi_bt = decile_backtest(filtered_with_toi, min_names=5)

for name, bt in [('unconditional', unconditional_bt),
                  ('vwap_dist + relative_strength filter', filtered_phase2_bt),
                  ('+ toi filter', filtered_with_toi_bt)]:
    if len(bt) == 0:
        print(f'{name}: 0 decisions (filter too restrictive to backtest)')
        continue
    gross_bps = bt['spread'].mean() * 1e4
    print(f'\n{name}: {len(bt):,} decisions, avg {bt["n"].mean():.1f} names/cross-section')
    print(f'  gross: {gross_bps:+.2f}bps')
    for cost in COST_BPS_GRID:
        print(f'  net at {cost}bps: {gross_bps - cost:+.2f}bps')


unconditional: 2,750 decisions, avg 23.0 names/cross-section
  gross: +5.91bps
  net at 0bps: +5.91bps
  net at 1bps: +4.91bps
  net at 2bps: +3.91bps
  net at 5bps: +0.91bps
  net at 10bps: -4.09bps

vwap_dist + relative_strength filter: 711 decisions, avg 6.3 names/cross-section
  gross: +6.96bps
  net at 0bps: +6.96bps
  net at 1bps: +5.96bps
  net at 2bps: +4.96bps
  net at 5bps: +1.96bps
  net at 10bps: -3.04bps

+ toi filter: 38 decisions, avg 5.4 names/cross-section
  gross: +8.94bps
  net at 0bps: +8.94bps
  net at 1bps: +7.94bps
  net at 2bps: +6.94bps
  net at 5bps: +3.94bps
  net at 10bps: -1.06bps


## Key findings

**The data-alignment problem from the first attempt is fully resolved.** Deriving every feature
directly from tick data (rather than merging with the separately-fetched, frozen minute-bar
cache) means all 126 tickers contribute, over their own full tick history, with nothing left to
misalign. This alone is real progress on the infrastructure question, independent of what the
numbers below say about trade imbalance itself.

**The measured effect is much larger than Phase 1/2 — and that itself is a reason for
skepticism, not celebration, until it's checked against an apples-to-apples sample.** Treat the
exact magnitudes below as provisional for two specific, named reasons:

1. **Different, shorter, more concentrated time window.** This sample is dominated by whichever
   few calendar days most tickers' tick fetches happened to land on (62 of 126 tickers on
   2026-10-19 alone — see the `fetch_day` distribution above), not a uniform one-month sample
   like Phase 1/2. If that window happened to include unusually active days (e.g. earnings
   season for some of these names), a larger measured reversal could reflect a short-lived
   regime rather than a persistently stronger effect.
2. **Trade density was checked and is NOT the obvious culprit.** A natural worry with
   tick-derived closes is mechanical bid-ask bounce inflating reversal on thin, few-trade
   minutes — but the median across all 126 tickers is tens of trades per minute (see the trade
   density cell above), which rules out "this is mostly empty-minute noise" as the explanation.
   It does not rule out bounce entirely, just the thinnest-liquidity version of that concern.

**Numbers, with that context in mind**:

- Unconditional IC: see the output above — several times larger than Phase 1/2's ~+0.05,
  consistent across `toi`, `vwap_dist`, and `relative_strength` buckets (all show real,
  monotonic-ish separation by tercile rather than one feature doing all the work the way
  `relative_strength` did in Phase 2).
- The combined filter and backtest numbers (see above) show whether adding `toi` to the Phase 2
  filter helps beyond `vwap_dist` + `relative_strength` alone, and whether the much larger IC
  translates into a backtest edge that survives more than Phase 1/2's ~1bps cost breakeven.

**Bottom line**: trade imbalance is worth taking seriously as a genuinely new feature — nothing
here suggests it's redundant with the existing price/volume features, unlike Garman-Klass
volatility in the Phase 2 addendum. But given how different this sample's time window is from
Phase 1/2's, the right next step before trusting the *magnitude* of the improvement is
re-running this same tick-derived-bars pipeline on a longer, more recent, and more uniformly-
dated tick history (the current cache's few-days-per-ticker spread was a side effect of how long
the original download took, not a deliberate sampling choice — see `STRATEGY.md`).

## Caveats

- **Sample window is short and unevenly distributed across tickers** — see Key Findings above.
  This is the single biggest reason to treat the magnitude of every number in this notebook as
  provisional, not a dismissal of the direction of the finding.
- **`volume_surprise` dropped** — needs a multi-session, minute-of-session baseline per ticker
  that ~1.5 days of tick history per ticker can't meaningfully support. Not tested here at all,
  rather than tested on a degenerate baseline.
- **Tick rule, not Lee-Ready** — same approximation the downloader deliberately chose; see
  `download_trade_ticks.py`'s docstring.
- **5 tickers skipped entirely** (`NVDA`, `MRVL`, `META`, `ORCL`, `AMZN`) — each hung
  reproducibly during its own tick fetch; see `SKIP_TICKERS`'s inline comments for the
  per-ticker diagnostic history.
- **Still requires shorting** for the backtest construct, same as Phases 1-2 — a research
  diagnostic, not a direct translation of the live (cash/long-only) account.
- **Multiple-comparisons risk**: three features bucketed three ways each, plus a combined
  filter and three backtest variants, all on one sample — the usual caution about not treating
  every individually-interesting number as independently confirmed applies here too.